# 01 — Data Preparation

Turn the network evidence (`236-XLMRat.pcap`) into clean, structured tables for
analysis. The pcap is parsed by `scripts/pcap_to_csv.py`, which writes **benign
metadata only** (timestamps, IPs, ports, sizes, HTTP request lines, DNS names) to
`data/processed/`. No packet payloads are written to disk.

If the CSVs are missing, regenerate them:
```bash
python -P scripts/pcap_to_csv.py evidence/xlmrat/236-XLMRat.pcap data/processed
```

In [1]:
import pandas as pd
pd.set_option('display.max_colwidth', 60)

packets = pd.read_csv('../data/processed/packets.csv', parse_dates=['time_utc'])
http    = pd.read_csv('../data/processed/http.csv')
dns     = pd.read_csv('../data/processed/dns.csv', parse_dates=['time_utc'])
print('packets:', packets.shape, '| http:', http.shape, '| dns:', dns.shape)

packets: (1550, 10) | http: (4, 12) | dns: (2, 7)


## Clean & standardize
De-duplicate, type the port columns, and confirm the capture window.

In [2]:
packets = packets.drop_duplicates(subset='frame').sort_values('frame').reset_index(drop=True)
for c in ['sport', 'dport']:
    packets[c] = pd.to_numeric(packets[c], errors='coerce').astype('Int64')
print('capture start:', packets.time_utc.min())
print('capture end  :', packets.time_utc.max())
print('duration (s) :', round((packets.time_utc.max() - packets.time_utc.min()).total_seconds(), 1))
packets.head()

capture start: 2024-01-09 17:27:27.576000
capture end  : 2024-01-09 17:40:13.947000
duration (s) : 766.4


,frame,time_utc,rel_time,src,dst,proto,sport,dport,length,tcp_flags
0,1,2024-01-09 17:27:27.576,0.000000,10.1.9.101,45.126.209.4,TCP,49708,222,66,S
1,2,2024-01-09 17:27:27.870,0.294054,45.126.209.4,10.1.9.101,TCP,222,49708,58,SA
2,3,2024-01-09 17:27:27.870,0.294591,10.1.9.101,45.126.209.4,TCP,49708,222,54,A
3,4,2024-01-09 17:27:27.871,0.295141,10.1.9.101,45.126.209.4,TCP,49708,222,357,PA
4,5,2024-01-09 17:27:27.871,0.295402,45.126.209.4,10.1.9.101,TCP,222,49708,54,A


## Hosts in scope

In [3]:
hosts = pd.concat([packets['src'], packets['dst']]).value_counts().rename('packets')
hosts.to_frame()

,packets
10.1.9.101,1550
45.126.209.4,1548
10.1.9.1,2


## HTTP and DNS evidence

In [4]:
display(http)
display(dns)

,frame,time_utc,rel_time,src,dst,dport,kind,method,host,uri,status,first_line
0,4,2024-01-09 17:27:27.871,0.295141,10.1.9.101,45.126.209.4,222,request,GET,45.126.209.4:222,/xlm.txt,NaN,GET /xlm.txt HTTP/1.1
1,6,2024-01-09 17:27:28.140,0.564806,45.126.209.4,10.1.9.101,49708,response,NaN,NaN,NaN,200.0,HTTP/1.1 200 OK
2,12,2024-01-09 17:27:29.161,1.585563,10.1.9.101,45.126.209.4,222,request,GET,45.126.209.4:222,/mdm.jpg,NaN,GET /mdm.jpg HTTP/1.1
3,14,2024-01-09 17:27:29.489,1.913001,45.126.209.4,10.1.9.101,49709,response,NaN,NaN,NaN,200.0,HTTP/1.1 200 OK


,frame,time_utc,src,dst,qname,qtype,is_response
0,350,2024-01-09 17:29:48.927,10.1.9.101,10.1.9.1,madmrx.duckdns.org,1,0
1,351,2024-01-09 17:29:49.262,10.1.9.1,10.1.9.101,madmrx.duckdns.org,1,1


**Prepared tables are ready.** Three IPs: victim `10.1.9.101`, gateway `10.1.9.1`,
and adversary `45.126.209.4`. Two HTTP GETs on port 222 and one DuckDNS lookup
stand out — carried forward into `02_security_eda`.